# Solution: Running Time-Series Diagnostics

You have two fitted models (ARIMA and SARIMAX) for electricity demand. Check whether their residuals are well-behaved and compare the models using information criteria.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy import stats

In [ ]:
import matplotlib as mplimport matplotlib.pyplot as plt# Udacity brand colorsUB = {"Brand Blue": "#175CFF", "Medium Blue": "#7BA2FF", "Light Blue": "#D2DFFF", "Navy": "#031643"}UN = {"Black": "#0A0B0F", "Dark Gray": "#5A5C62", "Medium Gray": "#9C9FA8", "Gray": "#CECFD4", "Light Gray": "#F2F2F2", "White": "#FFFFFF"}US = {"Orange": "#EE7622", "Yellow": "#F9DC5C", "Red": "#9C0D22", "Green": "#23CE6B"}# Thicker lines for visibilitympl.rcParams["lines.linewidth"] = 3mpl.rcParams["axes.linewidth"] = 2

In [ ]:
df = pd.read_csv("../data/electricity_demand.csv", parse_dates=["date"], index_col="date").asfreq("MS")
train_df = df.loc[:"2023-12-01"]
train_demand = train_df["demand_mwh"]
train_temp = train_df["avg_temp_f"]

arima_fit = SARIMAX(train_demand, order=(2, 1, 1)).fit(disp=False)
sarimax_fit = SARIMAX(train_demand, exog=train_temp, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12)).fit(disp=False)
print("Models fitted.")

## Task 1: Residual Diagnostics

Compute the Ljung-Box test at lag 12 and a normality test on the residuals. Return a dict with keys: `ljung_box_p`, `normality_p`, `aic`, `bic`.

In [ ]:
def run_residual_diagnostics(fitted):
    residuals = fitted.resid
    lb = acorr_ljungbox(residuals, lags=[12], return_df=True)
    _, normality_p = stats.normaltest(residuals.dropna())
    return {
        "ljung_box_p": lb["lb_pvalue"].values[0],
        "normality_p": normality_p,
        "aic": fitted.aic,
        "bic": fitted.bic,
    }

arima_diag = run_residual_diagnostics(arima_fit)
sarimax_diag = run_residual_diagnostics(sarimax_fit)

## Task 2: Print Comparison Table

Print AIC, BIC, Ljung-Box p-value, and normality p-value for both models in a formatted table.

In [ ]:
print("=== Model Comparison ===")
print(f"{'Model':<35} {'AIC':>10} {'BIC':>10} {'LB p':>10} {'Norm p':>10}")
print("-" * 77)
print(f"{'ARIMA(2,1,1)':<35} {arima_diag['aic']:>10.1f} {arima_diag['bic']:>10.1f} {arima_diag['ljung_box_p']:>10.4f} {arima_diag['normality_p']:>10.4f}")
print(f"{'SARIMAX(1,1,1)(1,1,1,12) + temp':<35} {sarimax_diag['aic']:>10.1f} {sarimax_diag['bic']:>10.1f} {sarimax_diag['ljung_box_p']:>10.4f} {sarimax_diag['normality_p']:>10.4f}")

## Task 3: Plot Residual ACF

Plot the ACF of residuals for both models side by side.

In [ ]:
def plot_residual_acf(arima_fit, sarimax_fit, lags=24):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    plot_acf(arima_fit.resid, ax=axes[0], lags=lags)
    axes[0].set_title("ARIMA(2,1,1) Residual ACF")
    plot_acf(sarimax_fit.resid, ax=axes[1], lags=lags)
    axes[1].set_title("SARIMAX(1,1,1)(1,1,1,12) Residual ACF")
    plt.tight_layout()
    plt.show()

plot_residual_acf(arima_fit, sarimax_fit)